# Solar Filament Segmentation 2026 ? Polygon Checkpoint, Union Decoder

This inference-only notebook evaluates and submits the polygon-trained YOLO26s checkpoint with the original union-mask decoder. It does not build YOLO training labels and does not call `model.train()`.

Attach the dataset containing the polygon run's `best.pt` checkpoint. The notebook finds a unique `best.pt` under `/kaggle/input`; if there are multiple checkpoints, set `CHECKPOINT_PATH` explicitly in the configuration cell.

The validation and submission decoder is: confidence filter ? union masks ? 8-connected components ? discard components below 20 pixels ? RLE. The fixed settings are confidence 0.25, NMS IoU 0.50, and 2048-pixel inference.


## 1. Install dependencies and import libraries

The training configuration uses disk caching rather than a RAM cache. Package installation disables pip's wheel cache to conserve the session disk.

In [ ]:
%pip -q install ultralytics pycocotools scikit-learn tqdm torchmetrics threadpoolctl psutil

In [ ]:
from pathlib import Path
import os
import csv
import gc
import random
import shutil
import time
import warnings
import multiprocessing
from concurrent.futures import ThreadPoolExecutor
from functools import partial

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
try:
    import psutil
except ImportError:
    psutil = None

from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm
from torchmetrics.segmentation import DiceScore
from threadpoolctl import threadpool_limits
from ultralytics import YOLO

# Hide standard Python warnings
warnings.filterwarnings("ignore")

# Hide underlying C-level / ctypes environment logs
os.environ["PYTHONWARNINGS"] = "ignore"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CUDA_DEVICE_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
MAX_GPUS_TO_USE = 2
GPU_DEVICE_IDS = list(range(min(CUDA_DEVICE_COUNT, MAX_GPUS_TO_USE)))
GPU_COUNT_USED = len(GPU_DEVICE_IDS)

if GPU_COUNT_USED >= 2:
    TRAIN_DEVICE = GPU_DEVICE_IDS
elif GPU_COUNT_USED == 1:
    TRAIN_DEVICE = GPU_DEVICE_IDS[0]
else:
    TRAIN_DEVICE = "cpu"

# Ultralytics prediction uses one explicit device; distributed training below
# uses both GPUs when they are available.
INFERENCE_DEVICE = GPU_DEVICE_IDS[0] if GPU_COUNT_USED else "cpu"

print(f"Training device(s): {TRAIN_DEVICE}")
print(f"Inference device: {INFERENCE_DEVICE}")
if CUDA_DEVICE_COUNT:
    for gpu_id in GPU_DEVICE_IDS:
        print(f"GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)}")
    print(f"CUDA devices available: {CUDA_DEVICE_COUNT}; using {GPU_COUNT_USED}")

CPU_COUNT = os.cpu_count() or 1
print(f"Logical CPU count: {CPU_COUNT}")

## 2. Configure inference

This notebook loads the already-trained polygon checkpoint. It keeps 2048-pixel inference and uses confidence 0.25, NMS IoU 0.50, and a 20-pixel connected-component cutoff.

In [ ]:
SEED = 42
VALIDATION_FRACTION = 0.2

IMAGE_SIZE = 2048
NUM_WORKERS = 4
WORKER_CPU_THREADS = 1
VAL_BATCH_SIZE = 1
MAX_PENDING_VALIDATION = NUM_WORKERS * 2
MAX_DETECTIONS = 100

RAM_LIMIT_GIB = 30.0
RAM_RESERVE_GIB = 6.0
GIB = 1024 ** 3

# Fixed post-processing recommended for the polygon checkpoint.
BEST_CONFIDENCE = 0.25
MIN_COMPONENT_AREA = 20
NMS_IOU = 0.50
CONFIDENCE_CANDIDATES = (0.15, 0.25, 0.35, 0.45, 0.55)
PQ_MATCH_IOU = 0.50

# Set an explicit path if the attached input dataset contains multiple best.pt files.
CHECKPOINT_PATH = None

SUBMISSION_PATH = Path.cwd() / "solar_filament_yolo26_polygon_union_submission.csv"
FINAL_METRICS_PATH = Path.cwd() / "solar_filament_yolo26_polygon_union_validation_metrics.csv"
PER_IMAGE_METRICS_PATH = Path.cwd() / "solar_filament_yolo26_polygon_union_per_image_metrics.csv"

CREATE_SUBMISSION = True


def _read_cgroup_memory_pair(limit_path, current_path):
    try:
        limit_text = Path(limit_path).read_text().strip()
        current_text = Path(current_path).read_text().strip()
        if limit_text == "max":
            return None
        limit_bytes = int(limit_text)
        current_bytes = int(current_text)
        if limit_bytes >= (1 << 60):
            return None
        return max(0, limit_bytes - current_bytes)
    except (OSError, ValueError):
        return None


def available_ram_bytes():
    for limit_path, current_path in (
        ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
        ("/sys/fs/cgroup/memory/memory.limit_in_bytes", "/sys/fs/cgroup/memory/memory.usage_in_bytes"),
    ):
        available = _read_cgroup_memory_pair(limit_path, current_path)
        if available is not None:
            return available
    if psutil is not None:
        try:
            return int(psutil.virtual_memory().available)
        except Exception:
            pass
    return None


def require_ram_headroom(stage, minimum_gib):
    available = available_ram_bytes()
    if available is None:
        print(f"[RAM] Could not read a runtime memory limit before {stage}; using bounded batches/workers.")
        return
    available_gib = available / GIB
    print(f"[RAM] Available before {stage}: {available_gib:.1f} GiB")
    if available_gib < minimum_gib:
        raise MemoryError(
            f"Only {available_gib:.1f} GiB RAM is available before {stage}; "
            f"need at least {minimum_gib:.1f} GiB under the {RAM_LIMIT_GIB:.1f} GiB session limit."
        )


def configure_main_process_threads():
    cv2.setNumThreads(WORKER_CPU_THREADS)
    try:
        torch.set_num_threads(WORKER_CPU_THREADS)
    except RuntimeError:
        pass


configure_main_process_threads()
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Image size: {IMAGE_SIZE} x {IMAGE_SIZE}")
print(f"Validation fraction: {VALIDATION_FRACTION:.0%}")
print(f"Confidence: {BEST_CONFIDENCE:.2f} | NMS IoU: {NMS_IOU:.2f}")
print(f"Minimum connected-component area: {MIN_COMPONENT_AREA} pixels")
print(f"CPU worker cap: {NUM_WORKERS}")


In [ ]:
def find_dataset_root():
    expected = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
    candidates = [Path.cwd(), Path.cwd().parent, Path("/kaggle/input")]

    for parent in candidates:
        if not parent.exists():
            continue
        for annotation in parent.glob("**/" + expected):
            root = annotation.parent.parent
            if (root / "train" / "train_images").is_dir() and (root / "test" / "test_images").is_dir():
                return root

    raise FileNotFoundError("Could not find MAGFiLO_1.0_Kaggle_2026 with train and test image folders.")

DATA_ROOT = find_dataset_root()
TRAIN_DIR = DATA_ROOT / "train"
TRAIN_IMAGE_DIR = TRAIN_DIR / "train_images"
TEST_IMAGE_DIR = DATA_ROOT / "test" / "test_images"
ANNOTATION_PATH = TRAIN_DIR / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

print(f"Dataset root: {DATA_ROOT}")
print(f"Annotation JSON: {ANNOTATION_PATH}")
print(f"Inference device: {INFERENCE_DEVICE}")
print(f"Validation fraction: {VALIDATION_FRACTION:.0%}")
print(f"PQ matching IoU threshold: {PQ_MATCH_IOU:.2f}")


## 3. Load annotations and create a leakage-safe split

The split is performed using the physical filename rather than the COCO record ID. Multiple annotation records for the same JPEG remain on one side of the split.

In [ ]:
coco = COCO(str(ANNOTATION_PATH))
image_ids = list(coco.imgs.keys())
records = pd.DataFrame([{"image_id": image_id, "file_name": coco.imgs[image_id]["file_name"]} for image_id in image_ids])

splitter = GroupShuffleSplit(n_splits=1, test_size=VALIDATION_FRACTION, random_state=SEED)
train_idx, val_idx = next(splitter.split(records, groups=records["file_name"]))

train_records = records.iloc[train_idx].reset_index(drop=True)
val_records = records.iloc[val_idx].reset_index(drop=True)

train_files = set(train_records["file_name"])
val_files = set(val_records["file_name"])

assert train_files.isdisjoint(val_files), "A physical image appears in both splits."

print(f"Training records: {len(train_records):,} across {len(train_files):,} physical images")
print(f"Validation records: {len(val_records):,} across {len(val_files):,} physical images")
print(f"Train/validation filename overlap: {len(train_files & val_files)}")

## 4. Configure bounded CPU workers

In [ ]:
def configure_worker_process():
    cv2.setNumThreads(WORKER_CPU_THREADS)
    try:
        torch.set_num_threads(WORKER_CPU_THREADS)
    except RuntimeError:
        pass
    try:
        torch.set_num_interop_threads(WORKER_CPU_THREADS)
    except RuntimeError:
        pass
    try:
        threadpool_limits(limits=WORKER_CPU_THREADS)
    except Exception:
        pass

    if hasattr(os, "sched_getaffinity"):
        available_cpus = sorted(os.sched_getaffinity(0))
        if available_cpus:
            identity = multiprocessing.current_process()._identity
            worker_index = identity[0] - 1 if identity else 0
            cpu = available_cpus[worker_index % len(available_cpus)]
            if hasattr(os, "sched_setaffinity"):
                os.sched_setaffinity(0, {cpu})


## 5. Load the polygon-trained checkpoint

The notebook discovers a unique `best.pt` under Kaggle inputs. No dataset rebuild or training run is performed.

In [ ]:
def find_checkpoint(explicit_path=None):
    if explicit_path:
        checkpoint = Path(explicit_path).expanduser()
        if not checkpoint.is_file():
            raise FileNotFoundError(f"Configured checkpoint does not exist: {checkpoint}")
        return checkpoint.resolve()

    roots = [Path("/kaggle/input"), Path.cwd()]
    candidates = set()
    for root in roots:
        if not root.exists():
            continue
        try:
            candidates.update(path.resolve() for path in root.rglob("best.pt") if path.is_file())
        except OSError:
            continue

    candidates = sorted(candidates)
    if not candidates:
        raise FileNotFoundError(
            "No best.pt checkpoint found. Attach the polygon-run model dataset or set CHECKPOINT_PATH."
        )
    if len(candidates) > 1:
        choices = "\n".join(str(path) for path in candidates)
        raise RuntimeError(
            "Found multiple best.pt checkpoints. Set CHECKPOINT_PATH to the polygon run's checkpoint:\n"
            + choices
        )
    return candidates[0]


require_ram_headroom("checkpoint load", minimum_gib=RAM_RESERVE_GIB)
best_checkpoint = find_checkpoint(CHECKPOINT_PATH)
model = YOLO(str(best_checkpoint))
print(f"Loaded polygon checkpoint: {best_checkpoint}")


## 6. Prediction and metric helpers

Validation and submission use the same decoder: YOLO instance masks ? confidence threshold 0.25 ? union ? 8-connected components ? minimum area 20 ? RLE. NMS IoU is fixed at 0.50.

In [ ]:
def to_yolo_image(image):
    if image.ndim != 2:
        raise ValueError(f"Expected grayscale image, got shape {image.shape}")
    return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

def get_test_image_paths():
    paths = [path for path in TEST_IMAGE_DIR.iterdir() if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg"}]
    return sorted(paths, key=lambda path: path.name)

def predict_instances_batch(images, min_confidence=0.15, nms_iou=NMS_IOU):
    sources = [to_yolo_image(image) for image in images]
    results = model.predict(
        source=sources, imgsz=IMAGE_SIZE, conf=min_confidence, iou=nms_iou, retina_masks=True,
        device=INFERENCE_DEVICE, batch=len(sources), max_det=MAX_DETECTIONS, verbose=False
    )
    outputs = []

    for image, result in zip(images, results):
        if result.masks is None or result.boxes is None:
            outputs.append([])
            continue

        masks = result.masks.data.detach().float().cpu().numpy()
        confidences = result.boxes.conf.detach().float().cpu().numpy()
        native_height, native_width = image.shape
        instances = []

        for mask, confidence in zip(masks, confidences):
            if mask.shape != image.shape:
                mask = cv2.resize(mask, (native_width, native_height), interpolation=cv2.INTER_NEAREST)
            instances.append({"mask": mask >= 0.5, "confidence": float(confidence)})
        outputs.append(instances)

    return outputs

def predict_instances(image, min_confidence=0.15, nms_iou=NMS_IOU):
    return predict_instances_batch([image], min_confidence=min_confidence, nms_iou=nms_iou)[0]

def union_instances(instances, confidence_threshold):
    selected = [instance for instance in instances if instance["confidence"] >= confidence_threshold]
    if not selected:
        return None

    foreground = np.zeros_like(selected[0]["mask"], dtype=bool)
    for instance in selected:
        foreground |= instance["mask"]
    return foreground

def split_into_filaments(binary_mask, min_area=MIN_COMPONENT_AREA):
    count, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask.astype(np.uint8), connectivity=8)
    components = []

    for label in range(1, count):
        area = int(stats[label, cv2.CC_STAT_AREA])
        if area < min_area:
            continue
        components.append(labels == label)

    def component_key(component):
        ys, xs = np.where(component)
        return (-int(component.sum()), int(ys.min()) if len(ys) else 0, int(xs.min()) if len(xs) else 0)

    return sorted(components, key=component_key)

def build_final_prediction_instances(instances, confidence_threshold):
    combined = union_instances(instances, confidence_threshold)
    if combined is None:
        return []
    return split_into_filaments(combined, MIN_COMPONENT_AREA)

def mask_to_rle(binary_mask):
    encoded = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    return encoded["counts"].decode("utf-8")


## 7. Full-resolution validation

Validation reports metrics for the confidence candidates and keeps 0.25 fixed for the submission. It evaluates the same union-and-connected-component decoder used for test images.

In [ ]:
dice_metric = DiceScore(num_classes=2, include_background=False, average="micro", input_format="index")

def calculate_pixel_metrics(prediction, target):
    prediction_tensor = torch.from_numpy(prediction.astype(np.int64)).unsqueeze(0)
    target_tensor = torch.from_numpy(target.astype(np.int64)).unsqueeze(0)

    prediction_has_foreground = bool(prediction.any())
    target_has_foreground = bool(target.any())

    if not prediction_has_foreground and not target_has_foreground:
        dice = 1.0
    else:
        dice = float(dice_metric(prediction_tensor, target_tensor).item())
        if np.isnan(dice):
            dice = 0.0

    intersection = int(np.logical_and(prediction, target).sum())
    union = int(np.logical_or(prediction, target).sum())
    true_positive = intersection

    false_positive = int(np.logical_and(prediction, ~target).sum())
    false_negative = int(np.logical_and(~prediction, target).sum())

    iou = intersection / union if union > 0 else 1.0
    precision_denominator = true_positive + false_positive
    recall_denominator = true_positive + false_negative

    precision = true_positive / precision_denominator if precision_denominator > 0 else 1.0
    recall = true_positive / recall_denominator if recall_denominator > 0 else 1.0

    return {
        "pixel_dice": float(dice),
        "pixel_iou": float(iou),
        "pixel_precision": float(precision),
        "pixel_recall": float(recall)
    }

def calculate_instance_metrics(prediction_instances, target_instances, iou_threshold=PQ_MATCH_IOU):
    num_predictions = len(prediction_instances)
    num_targets = len(target_instances)

    if num_predictions == 0 and num_targets == 0:
        return {"pq": 1.0, "sq": 1.0, "rq": 1.0, "tp_instances": 0, "fp_instances": 0, "fn_instances": 0, "one_to_many": 0, "many_to_one": 0, "matched_iou_sum": 0.0}
    if num_predictions == 0:
        return {"pq": 0.0, "sq": 0.0, "rq": 0.0, "tp_instances": 0, "fp_instances": 0, "fn_instances": num_targets, "one_to_many": 0, "many_to_one": 0, "matched_iou_sum": 0.0}
    if num_targets == 0:
        return {"pq": 0.0, "sq": 0.0, "rq": 0.0, "tp_instances": 0, "fp_instances": num_predictions, "fn_instances": 0, "one_to_many": 0, "many_to_one": 0, "matched_iou_sum": 0.0}

    iou_matrix = np.zeros((num_targets, num_predictions), dtype=np.float32)

    for gt_index, gt_mask in enumerate(target_instances):
        gt_area = int(gt_mask.sum())
        if gt_area == 0:
            continue
        for pred_index, pred_mask in enumerate(prediction_instances):
            pred_area = int(pred_mask.sum())
            if pred_area == 0:
                continue
            intersection = int(np.logical_and(gt_mask, pred_mask).sum())
            if intersection == 0:
                continue
            union = gt_area + pred_area - intersection
            iou_matrix[gt_index, pred_index] = intersection / union

    gt_relation_counts = (iou_matrix > iou_threshold).sum(axis=1)
    pred_relation_counts = (iou_matrix > iou_threshold).sum(axis=0)
    one_to_many = int((gt_relation_counts > 1).sum())
    many_to_one = int((pred_relation_counts > 1).sum())

    candidate_pairs = []
    for gt_index in range(num_targets):
        for pred_index in range(num_predictions):
            iou = float(iou_matrix[gt_index, pred_index])
            if iou > iou_threshold:
                candidate_pairs.append((iou, gt_index, pred_index))

    candidate_pairs.sort(key=lambda item: item[0], reverse=True)
    matched_gt, matched_predictions, matched_ious = set(), set(), []

    for iou, gt_index, pred_index in candidate_pairs:
        if gt_index in matched_gt or pred_index in matched_predictions:
            continue
        matched_gt.add(gt_index)
        matched_predictions.add(pred_index)
        matched_ious.append(iou)

    tp = len(matched_ious)
    fp = num_predictions - tp
    fn = num_targets - tp
    matched_iou_sum = float(np.sum(matched_ious))

    denominator = tp + 0.5 * fp + 0.5 * fn
    pq = matched_iou_sum / denominator if denominator > 0 else 0.0
    sq = matched_iou_sum / tp if tp > 0 else 0.0
    rq = tp / denominator if denominator > 0 else 0.0

    return {
        "pq": float(pq), "sq": float(sq), "rq": float(rq),
        "tp_instances": tp, "fp_instances": fp, "fn_instances": fn,
        "one_to_many": one_to_many, "many_to_one": many_to_one, "matched_iou_sum": matched_iou_sum
    }


In [ ]:
def load_validation_images(rows):
    images, valid_rows = [], []
    for row in rows:
        image_path = TRAIN_IMAGE_DIR / row["file_name"]
        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        if image is None:
            print(f"[WARNING] Could not read: {image_path}", flush=True)
            continue
        images.append(image)
        valid_rows.append(row)
    return valid_rows, images

def _worker_evaluate_validation_sample(task):
    configure_worker_process()
    global _local_val_coco

    if "_local_val_coco" not in globals():
        import sys
        old_stdout = sys.stdout
        sys.stdout = open(os.devnull, "w")
        _local_val_coco = COCO(task["annotation_path"])
        sys.stdout = old_stdout

    image_id = task["image_id"]
    file_name = task["file_name"]
    image_shape = tuple(task["image_shape"])
    polygons = task["polygons"]
    confidences = task["confidences"]

    target_instances = []
    annotation_ids = _local_val_coco.getAnnIds(imgIds=[image_id])
    annotations = _local_val_coco.loadAnns(annotation_ids)

    for annotation in annotations:
        mask = _local_val_coco.annToMask(annotation).astype(bool)
        if mask.any():
            target_instances.append(mask)

    target_union = np.zeros(image_shape, dtype=bool)
    for target_instance in target_instances:
        target_union |= target_instance

    outputs = {}
    for threshold in CONFIDENCE_CANDIDATES:
        prediction_union = np.zeros(image_shape, dtype=np.uint8)
        selected_predictions = 0

        for polygon, confidence in zip(polygons, confidences):
            if confidence < threshold:
                continue
            polygon = np.asarray(polygon, dtype=np.float32)
            if len(polygon) < 3:
                continue
            cv2.fillPoly(prediction_union, [np.round(polygon).astype(np.int32)], 1)
            selected_predictions += 1

        prediction_union = prediction_union.astype(bool)
        prediction_instances = split_into_filaments(prediction_union, MIN_COMPONENT_AREA)
        pixel_metrics = calculate_pixel_metrics(prediction_union, target_union)
        instance_metrics = calculate_instance_metrics(prediction_instances, target_instances)

        outputs[threshold] = {
            "file_name": file_name,
            "pixel_dice": pixel_metrics["pixel_dice"],
            "pixel_iou": pixel_metrics["pixel_iou"],
            "pixel_precision": pixel_metrics["pixel_precision"],
            "pixel_recall": pixel_metrics["pixel_recall"],
            "pq": instance_metrics["pq"],
            "sq": instance_metrics["sq"],
            "rq": instance_metrics["rq"],
            "tp_instances": instance_metrics["tp_instances"],
            "fp_instances": instance_metrics["fp_instances"],
            "fn_instances": instance_metrics["fn_instances"],
            "predicted_instances": len(prediction_instances),
            "ground_truth_instances": len(target_instances),
            "one_to_many": instance_metrics["one_to_many"],
            "many_to_one": instance_metrics["many_to_one"],
            "selected_predictions": selected_predictions
        }

    return outputs

In [ ]:
# Force garbage collection before allocating validation arrays
gc.collect()

require_ram_headroom("validation", minimum_gib=RAM_RESERVE_GIB + 2.0)
validation_start = time.perf_counter()

minimum_confidence = min(CONFIDENCE_CANDIDATES)

validation_examples = []

threshold_results = {
    threshold: {"per_image": []}
    for threshold in CONFIDENCE_CANDIDATES
}

val_rows = val_records[["image_id", "file_name"]].to_dict(orient="records")

total_validation_records = len(val_rows)

print("\n" + "=" * 90)
print("                 PARALLEL VALIDATION")
print("=" * 90)
print(f"Validation records : {total_validation_records:,}")
print(f"CPU workers        : {NUM_WORKERS}")
print(f"Target aggregate CPU usage: ~{NUM_WORKERS * 100}%")
print(f"GPU batch size     : {VAL_BATCH_SIZE}")
print("CPU workload       : polygon rasterization + connected components + metrics")
print("=" * 90, flush=True)

# Persistent CPU pool.
pool = multiprocessing.Pool(
    processes=NUM_WORKERS,
    initializer=configure_worker_process,
)

pending = []

try:
    for batch_start in range(0, total_validation_records, VAL_BATCH_SIZE):
        batch_rows = val_rows[batch_start:batch_start + VAL_BATCH_SIZE]

        loaded_rows, images = load_validation_images(batch_rows)

        if not images:
            continue

        # ------------------------------------------------------
        # GPU inference in batches.
        # ------------------------------------------------------

        batch_instances = predict_instances_batch(
            images,
            min_confidence=minimum_confidence,
        )

        # ------------------------------------------------------
        # Submit CPU evaluation asynchronously.
        # ------------------------------------------------------

        for row, image, instances in zip(loaded_rows, images, batch_instances):
            polygons = []
            confidences = []

            for instance in instances:
                mask = instance["mask"]

                contours, _ = cv2.findContours(
                    mask.astype(np.uint8),
                    cv2.RETR_EXTERNAL,
                    cv2.CHAIN_APPROX_SIMPLE,
                )

                if not contours:
                    continue

                contour = max(contours, key=cv2.contourArea)

                if len(contour) < 3:
                    continue

                polygon = contour.reshape(-1, 2).astype(np.float32)

                polygons.append(polygon)
                confidences.append(float(instance["confidence"]))

            task = {
                "annotation_path": str(ANNOTATION_PATH),
                "image_id": row["image_id"],
                "file_name": row["file_name"],
                "image_shape": image.shape,
                "polygons": polygons,
                "confidences": confidences,
            }

            pending.append(
                pool.apply_async(
                    _worker_evaluate_validation_sample,
                    (task,),
                )
            )

            if len(pending) >= MAX_PENDING_VALIDATION:
                result = pending.pop(0).get()

                for threshold in CONFIDENCE_CANDIDATES:
                    threshold_results[threshold]["per_image"].append(
                        result[threshold]
                    )

        # ------------------------------------------------------
        # Save a few examples for visualization.
        # ------------------------------------------------------

        # Save exactly 4 isolated examples for visualization without holding array references
        if len(validation_examples) < 4:
            for row, image, instances in zip(loaded_rows, images, batch_instances):
                if len(validation_examples) >= 4:
                    break

                annotations = coco.loadAnns(coco.getAnnIds(imgIds=[row["image_id"]]))
                target = np.zeros(image.shape, dtype=bool)

                for annotation in annotations:
                    target |= coco.annToMask(annotation).astype(bool)

                # Store detached copies to break cross-reference locks
                validation_examples.append(
                    {
                        "image": image.copy(),
                        "target": target.copy(),
                        "instances": [{k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in inst.items()} for inst in instances],
                        "file_name": row["file_name"],
                    }
                )

        processed = min(
            batch_start + len(loaded_rows),
            total_validation_records,
        )

        print(
            f"{processed:,}/{total_validation_records:,} records | "
            f"pending CPU jobs: {len(pending)}",
            flush=True,
        )

        try:
            del loaded_rows, images, batch_instances, polygons, confidences, task
        except NameError:
            pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(
        "\n[CPU] Draining remaining validation workers...",
        flush=True,
    )

    while pending:
        result = pending.pop(0).get()

        for threshold in CONFIDENCE_CANDIDATES:
            threshold_results[threshold]["per_image"].append(
                result[threshold]
            )

finally:
    pool.close()
    pool.join()

validation_time = time.perf_counter() - validation_start

threshold_rows = []

for threshold in CONFIDENCE_CANDIDATES:
    per_image_df = pd.DataFrame(
        threshold_results[threshold]["per_image"]
    )

    if per_image_df.empty:
        continue

    threshold_rows.append(
        {
            "confidence": threshold,
            "PQ": per_image_df["pq"].mean(),
            "SQ": per_image_df["sq"].mean(),
            "RQ": per_image_df["rq"].mean(),
            "Mean Dice": per_image_df["pixel_dice"].mean(),
            "Mean IoU": per_image_df["pixel_iou"].mean(),
            "Mean Precision": per_image_df["pixel_precision"].mean(),
            "Mean Recall": per_image_df["pixel_recall"].mean(),
            "TP": int(per_image_df["tp_instances"].sum()),
            "FP": int(per_image_df["fp_instances"].sum()),
            "FN": int(per_image_df["fn_instances"].sum()),
            "Predicted Instances": int(
                per_image_df["predicted_instances"].sum()
            ),
            "GT Instances": int(
                per_image_df["ground_truth_instances"].sum()
            ),
            "One-to-Many": int(per_image_df["one_to_many"].sum()),
            "Many-to-One": int(per_image_df["many_to_one"].sum()),
        }
    )

threshold_summary = (
    pd.DataFrame(threshold_rows)
    .sort_values("PQ", ascending=False)
    .reset_index(drop=True)
)

BEST_CONFIDENCE = 0.25
selected_validation_row = threshold_summary.loc[threshold_summary["confidence"] == BEST_CONFIDENCE].iloc[0]

print("\n" + "=" * 100)
print("                    VALIDATION RESULTS")
print("=" * 100)

display(threshold_summary.round(4))

print(f"\nSelected confidence threshold: {BEST_CONFIDENCE:.2f}")
print(f"Validation PQ at fixed confidence {BEST_CONFIDENCE:.2f}: {selected_validation_row['PQ']:.4f}")
print(f"Validation time: {validation_time:.1f}s")

print("=" * 100, flush=True)

## 8. Generate the competition submission

Test inference uses the loaded checkpoint, NMS IoU 0.50, confidence 0.25, union masks, connected components, and minimum component area 20. Rows are streamed to a new CSV.

In [ ]:
import queue

def _submission_worker_optimized(task, result_queue):
    """
    Worker process target. Processes the mask and pushes rows 
    straight to a thread-safe queue to avoid main-thread stalling.
    """
    combined_mask = task["combined_mask"]
    path_stem = task["path_stem"]
    components = split_into_filaments(combined_mask, MIN_COMPONENT_AREA)

    rows = []
    for filament_counter, component in enumerate(components, start=1):
        rows.append((f"{path_stem}_{filament_counter}", mask_to_rle(component)))
    
    # Push the result back to the main thread
    result_queue.put(rows)

def create_submission_parallel(output_path=SUBMISSION_PATH):
    paths = get_test_image_paths()
    if not paths:
        raise FileNotFoundError(f"No .jpg/.jpeg images found in {TEST_IMAGE_DIR}")

    require_ram_headroom("submission inference", minimum_gib=RAM_RESERVE_GIB + 2.0)
    total_images = len(paths)
    start_time = time.perf_counter()

    print(f"[START] Test images: {total_images:,}\n[CONFIG] GPU batch: {VAL_BATCH_SIZE}\n[CONFIG] CPU workers: {NUM_WORKERS}", flush=True)
    
    # Use a Manager to create a thread/process-safe Queue
    manager = multiprocessing.Manager()
    result_queue = manager.Queue()
    
    cpu_pool = multiprocessing.Pool(processes=NUM_WORKERS, initializer=configure_worker_process)

    with open(output_path, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["filament_id", "segmentation_rle"])

        total_rows_written = 0
        tasks_submitted = 0
        tasks_resolved = 0
        seen_filament_ids = set()

        try:
            for batch_start in range(0, total_images, VAL_BATCH_SIZE):
                batch_paths = paths[batch_start:batch_start + VAL_BATCH_SIZE]
                images = []

                for path in batch_paths:
                    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
                    if image is None:
                        raise OSError(f"Could not read test image: {path}")
                    images.append(image)

                # Heavy GPU inference step
                batch_instances = predict_instances_batch(images, min_confidence=BEST_CONFIDENCE)

                for path, image, instances in zip(batch_paths, images, batch_instances):
                    combined = union_instances(instances, BEST_CONFIDENCE)
                    if combined is None:
                        combined = np.zeros(image.shape, dtype=bool)

                    # Dispatch to pool completely asynchronously
                    cpu_pool.apply_async(
                        _submission_worker_optimized, 
                        args=({"combined_mask": combined, "path_stem": path.stem}, result_queue)
                    )
                    tasks_submitted += 1

                # Flush out whatever CPU work has finished up to this point WITHOUT blocking
                while True:
                    try:
                        worker_rows = result_queue.get_nowait()
                        for fid, rle in worker_rows:
                            if fid in seen_filament_ids:
                                raise ValueError(f"Duplicate filament ID detected: {fid}")
                            seen_filament_ids.add(fid)
                            writer.writerow([fid, rle])
                            total_rows_written += 1
                        tasks_resolved += 1
                    except queue.Empty:
                        break # No more completed tasks at this exact millisecond

                processed = min(batch_start + len(batch_paths), total_images)
                print(f"Inference Progress: {processed:,}/{total_images:,} | Active CPU Tasks: {tasks_submitted - tasks_resolved}", flush=True)
                
                del images, batch_instances
                gc.collect()

            # Clean up the pool so workers don't accept new tasks
            cpu_pool.close()
            
            # Block and drain the remaining finished items out of the queue
            print("\nFinishing remaining CPU post-processing tasks...", flush=True)
            while tasks_resolved < tasks_submitted:
                worker_rows = result_queue.get() # Blocking get is safe here because pool is closing
                for fid, rle in worker_rows:
                    if fid in seen_filament_ids:
                        raise ValueError(f"Duplicate filament ID detected: {fid}")
                    seen_filament_ids.add(fid)
                    writer.writerow([fid, rle])
                    total_rows_written += 1
                tasks_resolved += 1

            cpu_pool.join()

        finally:
            # Safeguard to ensure pool terminates if an exception throws early
            cpu_pool.close()
            cpu_pool.terminate()
            cpu_pool.join()

    elapsed = time.perf_counter() - start_time
    print(f"\n[SUCCESS] Submission complete!\nProcessed: {total_images:,} images\nFilament rows written: {total_rows_written:,}\nRuntime: {elapsed / 60:.2f} min\nSaved: {output_path}", flush=True)
    return pd.read_csv(output_path, nrows=5) if total_rows_written > 0 else None


## 9. Submission audit and validation report

In [ ]:
if CREATE_SUBMISSION:
    create_submission_parallel()
    
    audit_columns = None
    audit_row_count = 0
    audit_null_count = 0
    duplicate_ids = 0
    empty_rle_count = 0
    seen_ids = set()
    sample_rle_valid = False

    

    with open(SUBMISSION_PATH, newline="", encoding="utf-8") as audit_file:
        reader = csv.DictReader(audit_file)
        audit_columns = reader.fieldnames
        for row in reader:
            audit_row_count += 1
            if audit_row_count == 1:
                sample_rle_valid = bool(row.get("segmentation_rle"))

            if any(value is None for value in row.values()):
                audit_null_count += 1

            filament_id = row.get("filament_id")
            rle = row.get("segmentation_rle")
            if not filament_id or not rle:
                empty_rle_count += 1
            if filament_id in seen_ids:
                duplicate_ids += 1
            elif filament_id:
                seen_ids.add(filament_id)
else:
    audit_columns = ["filament_id", "segmentation_rle"]
    audit_row_count = 0
    audit_null_count = 0
    duplicate_ids = 0
    empty_rle_count = 0
    sample_rle_valid = False

print("--- Submission File Integrity Audit (streamed) ---")
print(f"Total rows: {audit_row_count:,}")
print(f"Null values: {audit_null_count:,}")
print(f"Duplicate filament IDs: {duplicate_ids:,}")
print(f"Empty IDs/RLE strings: {empty_rle_count:,}")
print(f"Sample RLE check: {sample_rle_valid}")

submission_valid = (
    set(audit_columns or []) == {"filament_id", "segmentation_rle"}
    and audit_null_count == 0
    and duplicate_ids == 0
    and empty_rle_count == 0
    and (sample_rle_valid or audit_row_count == 0)
)

print("\n" + ("[VERIFIED] Submission structure passed the local audit." if submission_valid else "[WARNING] Submission structure needs inspection."))

best_row = threshold_summary[threshold_summary["confidence"] == BEST_CONFIDENCE].iloc[0]

final_metrics = pd.DataFrame(
    {
        "Metric": [
            "Panoptic Quality (PQ)", "Segmentation Quality (SQ)",
            "Recognition Quality (RQ)", "Mean Dice", "Mean IoU",
            "Mean Precision", "Mean Recall", "True Positives",
            "False Positives", "False Negatives", "Predicted Instances",
            "Ground-Truth Instances", "One-to-Many (Fragmentation)",
            "Many-to-One (Over-Merging)", "Selected Confidence",
        ],
        "Value": [
            best_row["PQ"], best_row["SQ"], best_row["RQ"],
            best_row["Mean Dice"], best_row["Mean IoU"],
            best_row["Mean Precision"], best_row["Mean Recall"],
            int(best_row["TP"]), int(best_row["FP"]), int(best_row["FN"]),
            int(best_row["Predicted Instances"]), int(best_row["GT Instances"]),
            int(best_row["One-to-Many"]), int(best_row["Many-to-One"]),
            BEST_CONFIDENCE,
        ],
    }
)

print("\n" + "=" * 75)
print("                 FINAL VALIDATION REPORT")
print("=" * 75)
display(final_metrics.round(4))

selected_per_image = pd.DataFrame(threshold_results[BEST_CONFIDENCE]["per_image"])
distribution = (
    selected_per_image[["pixel_dice", "pixel_iou", "pq", "sq", "rq"]]
    .agg(["mean", "median", "std", "min", "max"])
    .T
)
print("\n--- Per-image metric distribution ---")
display(distribution.round(4))

diagnostics = pd.DataFrame(
    {
        "Diagnostic": [
            "Checkpoint", "NMS IoU", "Confidence", "Minimum component area",
            "Validation records", "Submission rows", "Submission valid",
        ],
        "Value": [
            str(best_checkpoint), NMS_IOU, BEST_CONFIDENCE, MIN_COMPONENT_AREA,
            len(selected_per_image), audit_row_count, submission_valid,
        ],
    }
)
print("\n--- Model / submission diagnostics ---")
display(diagnostics)

final_metrics.to_csv(FINAL_METRICS_PATH, index=False)
selected_per_image.to_csv(PER_IMAGE_METRICS_PATH, index=False)
print(f"Saved validation summary: {FINAL_METRICS_PATH}")
print(f"Saved per-image metrics: {PER_IMAGE_METRICS_PATH}")

## 10. Validation thresholds and examples

The chart shows validation PQ by confidence; the submitted prediction uses 0.25.

In [ ]:
fig, axis = plt.subplots(figsize=(7, 4.5))
axis.plot(threshold_summary["confidence"], threshold_summary["PQ"], marker="o", label="Validation PQ")
axis.axvline(BEST_CONFIDENCE, linestyle="--", label=f"Selected: {BEST_CONFIDENCE:.2f}")
axis.set(title="Validation PQ vs confidence threshold", xlabel="YOLO instance confidence", ylabel="Panoptic Quality (PQ)", ylim=(0, 1))
axis.grid(alpha=0.25)
axis.legend()
fig.tight_layout()
plt.show()

for example in validation_examples:
    image = example["image"]
    target = example["target"]
    instances = example["instances"]

    prediction = union_instances(instances, BEST_CONFIDENCE)
    if prediction is None:
        prediction = np.zeros_like(target, dtype=bool)

    selected_prediction_instances = build_final_prediction_instances(instances, BEST_CONFIDENCE)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    axes[0].imshow(image, cmap="gray")
    axes[0].set_title("Input image")
    axes[1].imshow(target, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Ground truth")
    axes[2].imshow(prediction, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title(f"Prediction · {len(selected_prediction_instances)} instance(s)")

    for axis in axes:
        axis.axis("off")

    fig.suptitle(example["file_name"])
    fig.tight_layout()
    plt.show()


## 11. Display a labeled prediction on an unseen test image

This visualization uses the same union-component decoder as the submission.

In [ ]:
def plot_labeled_prediction(image, instances, title):
    combined = union_instances(instances, BEST_CONFIDENCE)
    components = split_into_filaments(combined) if combined is not None else []

    fig, axis = plt.subplots(figsize=(9, 9))
    axis.imshow(image, cmap="gray")

    for filament_index, component in enumerate(components, start=1):
        axis.contour(component.astype(np.uint8), levels=[0.5], linewidths=1.2)
        moments = cv2.moments(component.astype(np.uint8))

        if moments["m00"]:
            center_x = moments["m10"] / moments["m00"]
            center_y = moments["m01"] / moments["m00"]
        else:
            ys, xs = np.where(component)
            center_x, center_y = xs.mean(), ys.mean()

        area = int(component.sum())
        axis.text(
            center_x, center_y, f"F{filament_index}\n{area:,} px", fontsize=8, ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.75)
        )

    axis.set_title(f"{title} · {len(components)} predicted filament(s)")
    axis.axis("off")
    fig.tight_layout()
    plt.show()

test_paths = get_test_image_paths()
if not test_paths:
    raise FileNotFoundError("No test images were found.")

test_path = test_paths[0]
test_image = cv2.imread(str(test_path), cv2.IMREAD_GRAYSCALE)
if test_image is None:
    raise OSError(f"Could not read test image: {test_path}")

test_instances = predict_instances(test_image, min_confidence=BEST_CONFIDENCE)
print(f"Displaying labeled prediction for test image: {test_path.name}")
plot_labeled_prediction(test_image, test_instances, f"Test prediction · {test_path.name} · confidence {BEST_CONFIDENCE:.2f}")

## 12. Outputs

- Loaded checkpoint: the polygon run's `best.pt` from the attached input dataset
- `solar_filament_yolo26_polygon_union_submission.csv` ? submission using union masks and connected components
- `solar_filament_yolo26_polygon_union_validation_metrics.csv` ? validation summary at confidence 0.25
- `solar_filament_yolo26_polygon_union_per_image_metrics.csv` ? per-image validation metrics

The notebook performs no training. The direct COCO-polygon checkpoint remains unchanged; only inference and post-processing are evaluated here.